# Self Retrievers

Un recuperador autoconsultante puede analizar y entender las consultas que se le hacen en lenguaje natural, y luego, puede buscar y filtrar información relevante de su base de datos o documentos almacenados basándose en esas consultas. Esto lo hace transformando las consultas en un formato estructurado que puede interpretar y procesar de manera eficiente. Esto significa que, además de comparar la consulta del usuario con los documentos para encontrar coincidencias, también puede filtrar los resultados según criterios específicos extraídos de la consulta del usuario.

![diagrama](../docs/media/diagrams/02.png)


## Librerías

In [2]:
from pprint import pprint

from dotenv import load_dotenv
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chat_models import ChatOpenAI
# from langchain.embeddings import OpenAIEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_mistralai import MistralAIEmbeddings
from langchain_classic.indexes import SQLRecordManager, index
from langchain_classic.retrievers import SelfQueryRetriever
from langchain_core.documents import Document
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from pydantic import BaseModel, Field

from src.langchain_docs_loader import LangchainDocsLoader, num_tokens_from_string

load_dotenv()

True

## Carga de datos

In [3]:
text_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=400,
    chunk_overlap=50,
    length_function=num_tokens_from_string,
)

loader = LangchainDocsLoader(include_output_cells=False)
docs = loader.load()
docs = text_splitter.split_documents(docs)
len(docs)

1459

In [6]:
docs = [doc for doc in docs if doc.page_content != "```"]

In [7]:
docs[0].metadata

{'source': 'https://docs.langchain.com/oss/python/common-errors',
 'title': 'Errors - Docs by LangChain',
 'language': 'en'}

## Inicializado de modelo de lenguaje

In [10]:
llm = ChatMistralAI(temperature=0.1, max_retries=2)

## Etiquetado de documentos

Los documentos por sí mismos son útiles, pero cuando son etiquetados con información adicional, pueden volverse más útiles. Por ejemplo, si etiquetamos los documentos con su idioma, podemos filtrar los documentos que no estén en el idioma que nos interesa. Si etiquetamos los documentos con su tema, podemos filtrar los documentos que no estén relacionados con el tema que nos interesa. De esta manera, podemos reducir el espacio de búsqueda y obtener mejores resultados.

### Creación de esquema de etiquetas

In [11]:
class Tags(BaseModel):
    completness: str = Field(
        description="Describes how useful is the text in terms of self-explanation.",
        json_schema_extra={
            "enum": ["Very", "Quite", "Medium", "Little", "Not"],
        },
    )
    code_snippet: bool = Field(
        default=False,
        description="Whether the text contains code snippet.",
    )
    description: bool = Field(
        default=False,
        description="Whether the text fragment includes a description.",
    )
    talks_about_vectorstore: bool = Field(
        default=False,
        description="Whether the text fragment talks about a vectorstore.",
    )
    talks_about_retriever: bool = Field(
        default=False,
        description="Whether the text fragment talks about a retriever.",
    )
    talks_about_chain: bool = Field(
        default=False,
        description="Whether the text fragment talks about a chain.",
    )
    talks_about_expression_language: bool = Field(
        default=False,
        description="Whether the text fragment talks about an langchain expression language.",
    )
    contains_markdown_table: bool = Field(
        default=False,
        description="Whether the text fragment contains markdown table.",
    )

pprint(Tags.model_json_schema())

{'properties': {'code_snippet': {'default': False,
                                 'description': 'Whether the text contains '
                                                'code snippet.',
                                 'title': 'Code Snippet',
                                 'type': 'boolean'},
                'completness': {'description': 'Describes how useful is the '
                                               'text in terms of '
                                               'self-explanation.',
                                'enum': ['Very',
                                         'Quite',
                                         'Medium',
                                         'Little',
                                         'Not'],
                                'title': 'Completness',
                                'type': 'string'},
                'contains_markdown_table': {'default': False,
                                            'description': 'Whet

### Creación de cadena de generación de etiquetas (etiquetador)

In [12]:
TAGGING_PROMPT = """Extract the desired information from the following passage.

Only extract the properties mentioned in the 'information_extraction' function.
Completness should involve more than one sentence.
To consider that a passage talks about a property, it is enough that it mentions it once.
If there is no mention of a property, set it to False. It only applies for the talk_about_* properties.

For instance,
To set `talks_about_vectorstore` to True, document should contain the word 'vectorstore' at least once.
To set `talks_about_retriever` to True, document should contain the word 'retriever' at least once.
To set `talks_about_chain` to True, document should contain the word 'chain' at least once.
To set `talks_about_expression_language` to True, document should contain the word 'expression language' or 'LCEL' at least once.

Passage:
{input}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", TAGGING_PROMPT),
    ("human", "{input}"),  # opcional, pero mantiene compatibilidad
])

structured_llm = llm.with_structured_output(
    Tags,
    # method="json_mode"             # alternativa si el proveedor lo soporta bien
)

tagging_chain = prompt | structured_llm

### Ejemplos de uso del etiquetador

Probablemente, un fragmento que únicamente contiene una lista de enlaces a otros fragmentos que también se encuentran indexados no es muy útil. Esto podría ocasionar que recuperemos un documento que no es relevante para la consulta, mientras el documento que sí es relevante no se encuentre en los primeros lugares de la lista de resultados.

In [ ]:
idx = 0

result = tagging_chain.invoke(input={"input": docs[idx].page_content})
print(result)

completness='Very' code_snippet=False description=True talks_about_vectorstore=False talks_about_retriever=False talks_about_chain=False talks_about_expression_language=False contains_markdown_table=True


Un fragmento con enlace a su documentación y ejemplo de uso sería más útil.

In [25]:
idx = 1000

result = tagging_chain.invoke(input={"input": docs[idx].page_content})
pprint(result)

Tags(completness='Little', code_snippet=False, description=False, talks_about_vectorstore=False, talks_about_retriever=False, talks_about_chain=False, talks_about_expression_language=False, contains_markdown_table=False)


In [28]:
idx = 1400

result = tagging_chain.invoke(input={"input": docs[idx].page_content})
pprint(result.model_dump())

{'code_snippet': False,
 'completness': 'Quite',
 'contains_markdown_table': False,
 'description': True,
 'talks_about_chain': False,
 'talks_about_expression_language': False,
 'talks_about_retriever': False,
 'talks_about_vectorstore': False}


### Etiquetado de documentos

In [13]:
tagging_results = tagging_chain.batch(
    inputs=[{"input": doc.page_content} for doc in docs[:200]],
    return_exceptions=True,
    config={
        "max_concurrency": 50,
    }
)

docs_with_tags = [
    Document(
        page_content=doc.page_content,
        metadata={
            **doc.metadata,
            **result.model_dump(),
        },
    )
    for doc, result in zip(docs, tagging_results)
    if not isinstance(result, Exception)
]

print(f"Documents with tags: {len(docs_with_tags)}")

Documents with tags: 60


In [30]:
docs_with_tags[0].metadata

{'source': 'https://docs.langchain.com/oss/python/common-errors',
 'title': 'Errors - Docs by LangChain',
 'language': 'en',
 'completness': 'Very',
 'code_snippet': False,
 'description': True,
 'talks_about_vectorstore': False,
 'talks_about_retriever': False,
 'talks_about_chain': False,
 'talks_about_expression_language': False,
 'contains_markdown_table': True}

## Indexado de documentos

In [14]:
vectorstore = Chroma(
    collection_name="langchain_docs",
    embedding_function=MistralAIEmbeddings(),
)

record_manager = SQLRecordManager(
    db_url="sqlite:///:memory:",
    namespace="chroma/langchain_docs",
)

record_manager.create_schema()

index(
    docs_source=docs_with_tags,
    key_encoder="sha256",
    record_manager=record_manager,
    vector_store=vectorstore,
    cleanup="full",
    source_id_key="source",
)

{'num_added': 60, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}

## Recuperación de documentos con un `Self Retriever`

### Creación de interfaz de los metadatos disponibles en el índice

In [15]:
metadata_field_info = [
    AttributeInfo(
        name="completness",
        description="Describes how useful is the text in terms of self-explanation.",
        type='enum["Very", "Quite", "Medium", "Little", "Not"]',
    ),
    AttributeInfo(
        name="code_snippet",
        description="Whether the text contains code snippet. Code snippets are valid markdown code blocks.",
        type="bool",
    ),
    AttributeInfo(
        name="description",
        description="Whether the text fragment includes a description.",
        type="bool",
    ),
    AttributeInfo(
        name="talks_about_vectorstore",
        description="Whether the text fragment talks about a vectorstore.",
        type="bool",
    ),
    AttributeInfo(
        name="talks_about_retriever",
        description="Whether the text fragment talks about a retriever.",
        type="bool",
    ),
    AttributeInfo(
        name="talks_about_chain",
        description="Whether the text fragment talks about a chain.",
        type="bool",
    ),
    AttributeInfo(
        name="talks_about_expression_language",
        description="Whether the text fragment talks about an langchain expression language.",
        type="bool",
    ),
    AttributeInfo(
        name="contains_markdown_table",
        description="Whether the text fragment contains markdown table.",
        type="bool",
    ),
]

document_content_description = "Langchain documentation"

### Creación de `retriever`

In [37]:
%%capture
!pip install lark

In [ ]:
llm = ChatMistralAI(temperature=0.0)
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    verbose=True,
)

### Recuperación de documentos con el `retriever`

In [ ]:
relevant_documents = retriever.invoke("contains code snippet")
relevant_documents

[Document(id='2b15e12fc40218355f3f90942c601f6b5b3159e46b21975ee6427646717a365d', metadata={'contains_markdown_table': False, 'talks_about_retriever': False, 'talks_about_chain': False, 'source': 'https://docs.langchain.com/oss/python/concepts/memory', 'code_snippet': True, 'description': True, 'talks_about_vectorstore': False, 'language': 'en', 'completness': 'Quite', 'talks_about_expression_language': False, 'title': 'Memory overview - Docs by LangChain'}, page_content='#### \u200bIn the background\n\nCreating memories as a separate background task offers several advantages. It eliminates latency in the primary application, separates application logic from memory management, and allows for more focused task completion by the agent. This approach also provides flexibility in timing memory creation to avoid redundant work.\n\nHowever, this method has its own challenges. Determining the frequency of memory writing becomes crucial, as infrequent updates may leave other threads without new

In [22]:
relevant_documents = retriever.invoke(
    "useful documents that talk about expression language"
)
relevant_documents

[Document(id='3446ea8b4857033e59f85e9ef8f16d8bbc01e785a775c008afb4511c82aa4656', metadata={'talks_about_retriever': False, 'description': True, 'talks_about_vectorstore': False, 'contains_markdown_table': True, 'code_snippet': False, 'title': 'Context overview - Docs by LangChain', 'source': 'https://docs.langchain.com/oss/python/concepts/context', 'language': 'en', 'talks_about_chain': False, 'completness': 'Very', 'talks_about_expression_language': True}, page_content='**Context engineering** is the practice of building dynamic systems that provide the right information and tools, in the right format, so that an AI application can accomplish a task. Context can be characterized along two key dimensions:\n\n1. By **mutability**:\n\n- **Static context**: Immutable data that doesn’t change during execution (e.g., user metadata, database connections, tools)\n- **Dynamic context**: Mutable data that evolves as the application runs (e.g., conversation history, intermediate results, tool ca